In [24]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [26]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None
load_in_4bit = False

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-3B",
    max_seq_length = max_seq_length,
    dtype = dtype,
)

==((====))==  Unsloth 2025.4.7: Fast Qwen2 patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [32]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

from datasets import load_dataset
dataset = load_dataset("Vezora/Tested-143k-Python-Alpaca", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

In [37]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 143,327 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568/3,000,000,000 (1.00% trained)


Step,Training Loss
1,0.878500
2,1.109000
3,0.938700
4,0.871300
5,0.811400
6,0.672000
7,0.538300
8,0.738000
9,0.864000
10,0.849000


In [42]:
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    alpaca_prompt.format(
        """In the realm of cryptography, the security of encryption algorithms is paramount. However, side-channel attacks, such as timing attacks, can compromise the security of an encryption system. A constant-time comparison function is a fundamental component of cryptographic protocols that ensures that the execution time of the comparison does not depend on the values being compared, making it resistant to timing attacks.
        Consider a scenario where you are implementing a secure token-based authentication system in a Python application. The system should use a cryptographic hash function to ensure that the tokens are resistant to tampering. However, you must also ensure that the system is resistant to timing attacks.
        1. Implement a Python function that performs a constant-time string comparison. This function should take two strings as input and return a boolean indicating whether they are equal. The comparison should be performed in a way that does not leak information about the length of the strings or the time taken to compare them.
        2. Explain how your implementation of the constant-time comparison function ensures that the execution time is independent of the strings being compared.
        3. Using your constant-time comparison function, write a Python function that simulates a simple token-based authentication system. The function should take a user's username and a token as inputs, check the token against a stored hash of the correct token (which should be known to the system and not be vulnerable to timing attacks), and return a boolean indicating whether the authentication was successful.
        Provide the Python code for the constant-time string comparison function and the token-based authentication system function.
        Note: This question assumes a basic understanding of cryptographic hash functions, Python programming, and the principles of constant-time execution. It is intended for an advanced level of knowledge in cybersecurity and programming.""", # instruction
        "", # input
        "", # output
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 1024)

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
In the realm of cryptography, the security of encryption algorithms is paramount. However, side-channel attacks, such as timing attacks, can compromise the security of an encryption system. A constant-time comparison function is a fundamental component of cryptographic protocols that ensures that the execution time of the comparison does not depend on the values being compared, making it resistant to timing attacks.
        Consider a scenario where you are implementing a secure token-based authentication system in a Python application. The system should use a cryptographic hash function to ensure that the tokens are resistant to tampering. However, you must also ensure that the system is resistant to timing attacks.
        1. Implement a Python function that performs a constant-time string comparison. This func

In [ ]:
model.save_pretrained("Qwen2.5-Coder-3B-143k-Python-Alpaca_model")
tokenizer.save_pretrained("Qwen2.5-Coder-3B-143k-Python-Alpaca_model")
model.push_to_hub("Vishu06/Qwen2.5-Coder-3B-143k-Python-Alpaca_model", token = "")
tokenizer.push_to_hub("Vishu06/Qwen2.5-Coder-3B-143k-Python-Alpaca_model", token = "")

In [ ]:
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Write a python program for RSA Algorithm?", # instruction
        "", # input
        "", # output
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 256)

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Write a python program for RSA Algorithm?

### Input:


### Response:
Here is a simple implementation of the RSA algorithm in Python:

```python
import random

def is_prime(n):
    if n <= 1:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    for i in range(3, int(n**0.5) + 1, 2):
        if n % i == 0:
            return False
    return True

def generate_prime(bits):
    while True:
        n = random.randint(2**(bits-1), 2**bits - 1)
        if is_prime(n):
            return n

def egcd(a, b):
    if a == 0:
        return (b, 0, 1)
    else:
        g, y, x = egcd(b % a, a)
        return (g, x - (b // a) * y, y)

def modinv(a, m):
    g, x, y = egcd(a, m)
    if g != 1:
        raise Exception('modular inverse does not exist')
    else:
        return x % m
